<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

# Unsupervised learning exercise: Clustering load profiles of different users

</div>

**Objective:** You work at a distribution system operator (DSO). Smart meters give you the hourly net load of every building, but not always which buildings have rooftop PV, an electric vehicle (EV) or a heat pump (HP): PV is usually registered, EVs and heat pumps often are not, and all three change how the grid is loaded. Can you find them from the load alone?

The dataset has 700 Belgian meters (2024) in four groups of 175: `Nothing`, `PV`, `EV_PV` (EV + PV) and `EV_HP` (EV + heat pump, no PV). The labels are never used to build the clusters; we use them only afterwards, to measure how well each clustering recovers them. The exercise follows Canaydin, Depoortere & Kazmi (ISGT Europe 2024), *Feature-based identification of DERs in buildings from aggregated smart meter data* (PDF in the course folder).

| Part | How | Time |
|---|---|---|
| 0. Load and explore the data | together, fill the blanks | 10 min |
| 1. Clustering the average daily profiles | you, then solution | 15 min |
| 2. Comparing curves with Dynamic Time Warping (DTW) | together, fill the blanks | 10 min |
| 3. Clustering statistical features | you (fill the blanks), then solution | 15 min |
| 4. Choosing k and interpreting the clusters | you, then solution | 10 min |
| 5. Another model: Gaussian Mixture (GMM) | together, fill the blanks | 10 min |
| 6. Visualisation: PCA and t-SNE | PCA: you, then solution; t-SNE: together | 15 min |
| 7. Conclusions | together | 5 min |

The data is stored as Parquet (22 MB, a random sample of 175 meters per class from the original 1,200). Reading it needs `pyarrow`, and Section 2 needs `tslearn`: if the first code cell fails, run `pip install pyarrow tslearn`.

Answer the ✏️ questions in the cell below each one.

**How to work through this notebook:** each section says how it runs. Code cells that contain only `#` comments are for you to write; replace every `___` by the right value or name (one blank = one short expression). Answer the ✏️ questions in the markdown cells below them, and fill the table in the conclusions with your own results.

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 0. Load and explore the data

</div>

*Together: fill in the blanks as we go.*

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score, accuracy_score, f1_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore', message='h5py not installed')   # tslearn's optional HDF5 support, not used here
from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import dtw

sns.set(style="whitegrid")

df_raw = pd.___('Data/sm_hourly_profiles_belgium_700.parquet')
df_raw

**What do the columns mean?**

* **[SM_ID]**: smart meter ID (SM_001 to SM_700)
* **[Label]**: installed DERs (hidden during clustering)
* **[Start_Date]**: start of the hour, Belgian local time (CET/CEST). On 31 March 02:00 does not exist (clock change), so each meter has 8,783 hours instead of 8,784.
* **[SM_kWh]**: net energy in the hour [kWh] (= average power in kW). **Negative = export** to the grid (PV).

In [ ]:
print(df_raw.___)
print(df_raw.drop_duplicates(___)['Label'].___())
df_raw.___()

✏️ **Q0.** (1) How many rows does the table have, and how many hours does that make per meter? (2) How many meters are there per class? (3) What range does `SM_kWh` cover, what does a negative value mean, and why is the median so far below the mean?

*Your answer:*

Before clustering, look at what we are trying to recover. The plot below uses the labels, which is fine for exploration: it shows the differences a good clustering would have to pick up. Each class is averaged by hour of the day and season, as in Fig. 1 of the paper. First add three helper columns: the hour of the day, the month, and the season (winter = December to February, summer = June to August, the rest spring/autumn).

In [ ]:
df_raw['hour'] = df_raw['Start_Date'].dt.___
df_raw['month'] = df_raw['Start_Date'].dt.___
df_raw['season'] = np.where(df_raw['month'].isin([___]), 'Winter',
                   np.where(df_raw['month'].isin([___]), 'Summer', 'Spring/Autumn'))

In [ ]:
# Mean net load of each class at each hour, per season: the curves a good clustering should tell apart
profile = df_raw.groupby(['season', 'Label', 'hour'])['SM_kWh'].mean().reset_index()
# One panel per season, one line per class (labels used for exploration only, never for clustering)
sns.relplot(data=profile, x='hour', y='SM_kWh', hue='Label', col='season',
            col_order=['Winter', 'Spring/Autumn', 'Summer'], kind='line', height=3.2)
plt.show()

> **Reading the plot:** PV is the strongest signature: the net load of `PV` and `EV_PV` goes negative around midday, down to about −1.5 kWh in summer but barely to zero in winter. `EV_HP` is the largest consumer all year, highest in winter (heat pump) and peaking around midnight (night charging). `PV` and `EV_PV` have the same shape and differ mostly in the evening and at night, when the EV is charged: expect the clustering to confuse these two.

In [ ]:
# From now on we only use the load. The labels are kept aside, aligned by SM_ID, to validate the clusters.
df_sm = df_raw[['SM_ID', 'hour', 'SM_kWh']]
labels = df_raw.groupby('SM_ID')['Label'].first()   # one label per meter, sorted by SM_ID
labels.head()

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 1. Clustering the average daily profiles

</div>

*You write the code; then we go over the solution together.*

The slide *Clustering time series* gave two options for curves like these: cluster the raw values, or derive features first. We start with the raw values. Each meter becomes its average daily profile, 24 numbers: the mean net load at 0 h, 1 h, …, 23 h over the whole year. Averaging 366 days removes day-to-day noise, but also the seasons: the heat pump's winter load and the deep summer PV dip are blended into one curve.

In [ ]:
# Build df_profiles: one row per meter (SM_ID), one column per hour (0-23), values = mean SM_kWh over the year.
# Hint: group df_sm by ['SM_ID', 'hour'], take the mean of 'SM_kWh', then .unstack().



All 24 columns share a unit (kWh), so standardising them is optional. We leave them as they are, so that the hours in which meters differ most (around midnight and midday) weigh more in the distance than the quieter morning hours. We set k = 4 because we know there are four classes; Section 4 asks what you would do without that knowledge.

**Comparing clusters with labels: the Adjusted Rand Index (ARI).** Accuracy or F1 would need each cluster to be matched to a class first, and cluster numbers are arbitrary. The ARI compares the two groupings directly, through pairs of meters: a pair is an agreement when the clustering and the labels both put the two meters together, or both keep them apart. The share of agreeing pairs is the Rand index (RI), which is misleading on its own: with four groups of 175, a random clustering already agrees on about 62 % of pairs, because most pairs are apart in both. The ARI rescales it so that chance gives 0:

$$\text{ARI} = \frac{\text{RI} - \mathbb{E}[\text{RI}]}{\max(\text{RI}) - \mathbb{E}[\text{RI}]}$$

$\mathbb{E}[\text{RI}]$ is the RI expected from a random clustering with the same cluster sizes (≈ 0.62 here) and $\max(\text{RI})$ its best possible value. ARI = 1 means identical groupings, ≈ 0 means no better than chance, and it can be slightly negative. Since it only uses the crosstab, the numbering of the clusters does not matter.

The ARI has no counterpart in classification, so we also report a more familiar view. Name each cluster after the label carried by most of its meters, and the clustering becomes a classifier that can be scored with accuracy and macro-F1. Two clusters may get the same name; the class left without a cluster then has no correct predictions, which macro-F1 penalises and accuracy hides. The naming uses the labels, so this is an evaluation step only, not something a DSO could do.

In [ ]:
# Run KMeans(n_clusters=4, n_init=10, random_state=0) on df_profiles and store the fitted model in km_raw.
# Print the ARI (adjusted_rand_score), the accuracy and the macro-F1, and show pd.crosstab(labels, km_raw.labels_).
# To name each cluster after its majority label: pred = pd.crosstab(labels, km_raw.labels_).idxmax().loc[km_raw.labels_].values



Look at where `EV_HP` ends up in the crosstab. Euclidean distance on raw kWh is dominated by how much a meter consumes, and `EV_HP` meters range from moderate to very large. To compare *shapes* instead, rescale each meter's profile (each row) to mean 0 and standard deviation 1.

In [ ]:
# Normalise each row of df_profiles (z-score per meter) into df_shape, run k-means (k=4, n_init=10, random_state=0) into km_shape,
# and print the ARI, accuracy, macro-F1 and the crosstab, as above.
# Hint: df_profiles.sub(df_profiles.mean(axis=1), axis=0) subtracts each row's mean; .div(df_profiles.std(axis=1), axis=0) divides by each row's std.
# (StandardScaler would scale each column, i.e. each hour, which is not what we want here.)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(km_raw.cluster_centers_.T)
axes[0].set_title('Centroids, raw profiles [kWh]')
axes[1].plot(km_shape.cluster_centers_.T)
axes[1].set_title('Centroids, normalised profiles')
for ax in axes:
    ax.set_xlabel('hour'); ax.legend(range(4), title='cluster')
plt.show()

✏️ **Q1.** Name the four raw-profile clusters from their centroids. Normalising was meant to remove the size effect, yet the ARI went down: what information did it throw away, and which classes does that mix up?

*Your answer:*

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 2. Comparing curves with Dynamic Time Warping (DTW)

</div>

*Together: fill in the blanks as we go.*

The clustering of Section 1 used the Euclidean distance, which compares two curves hour by hour. Two homes that charge the same EV for 1 hour at 21 h and at 23 h then look as different as a home with an EV and one without (or even more). DTW first aligns the two curves: each hour of one profile may be matched with a nearby hour of the other, and the distance is summed along the best alignment, the *warping path*. Without a limit, the alignment can stretch a curve so far that a midnight peak is matched with the midday PV dip; a window of ±2 h (the Sakoe–Chiba band) keeps the matching physically plausible.

<img src="Figures/dtw_vs_euc.svg" width="900">

On the left, the Euclidean distance matches each time step with the same time step of the other curve, so two curves with the same shape, shifted in time, end up far apart. On the right, DTW matches each point with its best counterpart nearby: the grey lines lean, and the two bumps are matched with each other. The same in numbers, for two identical evening peaks two hours apart:

In [ ]:
hours = np.arange(24)
peak_19 = np.exp(-(hours - 19) ** 2 / 2)   # evening peak at 19 h
peak_21 = np.exp(-(hours - 21) ** 2 / 2)   # the same peak two hours later

print('Euclidean distance:', round(np.linalg.norm(peak_19 - peak_21), 2))
print('DTW distance, window ±2 h:', round(dtw(peak_19, peak_21, global_constraint=___, sakoe_chiba_radius=___), 2))

Now cluster the average daily profiles of Section 1 with DTW. The centroids are no longer plain means but DTW averages (DBA), which is slower: expect about 10 s per run.

In [ ]:
X_series = df_profiles.values[:, :, np.newaxis]   # tslearn expects (meters, hours, 1)
km_dtw = TimeSeriesKMeans(n_clusters=4, metric=___, n_init=3, max_iter=30, random_state=0,
                          metric_params={'global_constraint': 'sakoe_chiba', 'sakoe_chiba_radius': ___}).fit(X_series)

pred = pd.crosstab(labels, km_dtw.labels_).idxmax().loc[km_dtw.labels_].values
print('ARI DTW:', round(adjusted_rand_score(labels, ___), 3))
print('Accuracy:', round(accuracy_score(labels, pred), 3), '| macro-F1:', round(f1_score(labels, pred, average='macro'), 3))
pd.crosstab(labels, km_dtw.labels_)

In [ ]:
# Is the result stable? Repeat with other random seeds (k-means on the same profiles with the Euclidean distance varies by about 0.01)
for seed in [1, 2]:
    y_seed = TimeSeriesKMeans(n_clusters=4, metric='dtw', n_init=3, max_iter=30, random_state=___,
                              metric_params={'global_constraint': 'sakoe_chiba', 'sakoe_chiba_radius': 2}).fit_predict(X_series)
    print('seed', seed, '| ARI DTW:', round(adjusted_rand_score(labels, y_seed), 3))

✏️ **Q2.** Compare the DTW clustering with the Euclidean one of Section 1 (ARI, crosstab, other seeds). Does DTW help, which mix-up does it reduce, and why can it not fix the others?

*Your answer:*

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 3. Clustering statistical features

</div>

*You fill in the blanks; then we go over the solution together.*

Now the other option from the slide: summarise each meter with a few numbers computed over the whole year, one row per meter and one column per feature. We start with the statistics anyone would compute first, without any knowledge of DERs.

In [ ]:
df_features = df_sm.groupby('SM_ID')['SM_kWh'].agg(['mean', ___, 'std', 'sum'])
df_features[['mean', 'sum']].corr()

✏️ **Q3.** Why are `mean` and `sum` perfectly correlated? Would standardising the features remove the problem?

*Your answer:*

In [ ]:
df_features = df_features.drop(columns=___)
# Features have different units and ranges: standardise them before k-means
scaler = StandardScaler()
X_scaled = scaler.___(df_features)

kmeans = KMeans(n_clusters=___, n_init=10, random_state=0).fit(X_scaled)
pred = pd.crosstab(labels, kmeans.labels_).idxmax().loc[kmeans.labels_].values
print('ARI mean/median/std:', round(adjusted_rand_score(labels, kmeans.labels_), 3))
print('Accuracy:', round(accuracy_score(labels, pred), 3), '| macro-F1:', round(f1_score(labels, pred, average='macro'), 3))
pd.crosstab(labels, kmeans.labels_)

With 3 numbers per meter instead of 24, the ARI is 0.40, close to the raw profiles (0.35), and k-means fails in the same way: `PV` and `EV_PV` share a cluster, and `EV_HP` is split in two. A different representation, a similar outcome, with far fewer features. Next we add three more statistics: the yearly minimum and maximum of the net load, and the share of hours with export.

In [ ]:
df_features['min'] = df_sm.groupby('SM_ID')['SM_kWh'].___()
df_features['max'] = df_sm.groupby('SM_ID')['SM_kWh'].___()
df_features['neg_share'] = (df_sm['SM_kWh'] ___).groupby(df_sm['SM_ID']).mean()   # the mean of True/False is a share
X_scaled = scaler.fit_transform(df_features)

kmeans = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X_scaled)
pred = pd.crosstab(labels, kmeans.labels_).idxmax().loc[kmeans.labels_].values
print('ARI + min/max/neg_share:', round(adjusted_rand_score(labels, kmeans.labels_), 3))
print('Accuracy:', round(accuracy_score(labels, pred), 3), '| macro-F1:', round(f1_score(labels, pred, average='macro'), 3))
pd.crosstab(labels, kmeans.labels_)

✏️ **Q4.** The ARI rises from 0.40 to 0.50. Which DER does each new feature respond to, and which mix-up in the crosstab does it fix?

*Your answer:*

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 4. Choosing k and interpreting the clusters

</div>

*You write the code; then we go over the solution together.*

So far we have set k = 4 from the labels. The DSO in the objective would not know how many groups to look for. Compute the elbow (WCSS) and the silhouette for k = 2 to 10 on the six features of Section 3. Both measure how compact and well separated the clusters are in feature space, which is not the same question as how many kinds of DER exist.

In [ ]:
# For k = 2 to 10, fit KMeans(n_clusters=k, n_init=10, random_state=0) on X_scaled.
# Store each model's inertia_ in a list wcss and silhouette_score(X_scaled, model.labels_) in a list sil.
# Plot wcss and sil against k side by side (as in the customer segmentation notebook).



In [ ]:
# Without labels, we interpret the clusters through their centres. Put the k = 4 centres (kmeans.cluster_centers_)
# back into the original units with scaler.inverse_transform, as a DataFrame with the feature names as columns.



✏️ **Q5.** Which k would you choose from these two plots alone, and why? Then, using the centres only (no labels), name the four clusters as you would report them to the DSO.

*Your answer:*

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 5. Another model: Gaussian Mixture (GMM)

</div>

*Together: fill in the blanks as we go.*

Sections 3 and 4 showed the same failure twice: k-means splits `EV_HP` and merges `PV` with `EV_PV`. This follows from how k-means assigns points: each meter joins its nearest centre, which implicitly assumes round clusters of similar spread. After scaling, `EV_HP` is two to four times as spread out as `Nothing` depending on the feature, so a single centre cannot hold it. A Gaussian Mixture Model fits one Gaussian per cluster with its own covariance, so clusters can be elongated and of different sizes. Same features, same k: only the model changes.

In [ ]:
gmm = GaussianMixture(n_components=___, n_init=3, random_state=0).fit(X_scaled)
y_gmm = gmm.___(X_scaled)
pred = pd.crosstab(labels, y_gmm).idxmax().loc[y_gmm].values
print('ARI GMM:', round(adjusted_rand_score(labels, ___), 3))
print('Accuracy:', round(accuracy_score(labels, pred), 3), '| macro-F1:', round(f1_score(labels, pred, average='macro'), 3))
pd.crosstab(labels, y_gmm)

✏️ **Q6.** With the same features, the ARI goes from 0.50 (k-means) to 0.73 (GMM). Compare the two crosstabs: which classes does the GMM separate that k-means could not, and what property of those classes explains it?

*Your answer:*

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 6. Visualisation: PCA and t-SNE

</div>

*PCA: you write the code, then we go over the solution together. t-SNE: together, fill in the blanks as we go.*

PCA gives a second, label-free view of the same meters. We apply it to both representations: the average daily profiles of Section 1, whose loadings can be read hour by hour, and the six features of Section 3 on which the GMM was fitted. As in the slides, we standardise first. For the six features this is required (kWh and shares on different scales); for the profiles all columns are kWh, and standardising changes little because every hour has a similar spread across meters (0.5 to 1.0 kWh).

In [ ]:
# Standardise df_profiles with StandardScaler into scaled_data and fit PCA(n_components=5) on it (pca_profiles).
# Plot side by side: the explained variance ratio of the 5 components (bar chart) and the loadings of PC1 and PC2 against the hour
# (pca_profiles.components_[0] and [1]).



In [ ]:
# Fit PCA() on X_scaled (pca_features) and store the projected meters in results_pca (pca_features.transform).
# Print the explained variance ratio and show the PC1 and PC2 loadings of the six features as a DataFrame.



In [ ]:
labels_encoded, label_names = pd.factorize(labels)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(results_pca[:, 0], results_pca[:, 1], c=y_gmm, s=15, alpha=0.5, cmap='viridis')
axes[0].set_title('PCA of the features, coloured by GMM cluster')
sc = axes[1].scatter(results_pca[:, 0], results_pca[:, 1], c=labels_encoded, s=15, alpha=0.5, cmap='viridis')
axes[1].legend(sc.legend_elements()[0], label_names, title='Label'); axes[1].set_title('PCA of the features, coloured by hidden label')
for ax in axes:
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
plt.show()

✏️ **Q7.** Read the loadings of both PCAs. What does each first and second component represent physically, and on what do the two views agree?

*Your answer:*

In [ ]:
X_embedded = TSNE(n_components=___, perplexity=___, init='pca', random_state=42).___(X_scaled)

plt.figure(figsize=(7, 5))
sc = plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=labels_encoded, s=15, cmap='viridis')
plt.legend(sc.legend_elements()[0], label_names, title='Label')
plt.title('t-SNE of the statistical features'); plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.show()

> **Caution:** t-SNE is for visualisation only. The map agrees with the crosstabs: `PV` and `EV_PV` share one region, while `Nothing` and `EV_HP` form a second one in which each class keeps its own end. But t-SNE only preserves neighbourhoods: the distances between islands and their sizes carry no meaning, and the picture changes with the perplexity and the random seed. Use it to look at the data, not to cluster it or to measure how separable the classes are.

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## 7. Conclusions

</div>

| Approach | Features | Model | ARI | Accuracy | Macro-F1 |
|---|---|---|---|---|---|
| Average daily profiles | 24 hours (kWh) | k-means | ___ | ___ | ___ |
| Average daily profiles | 24 hours (shape) | k-means | ___ | ___ | ___ |
| Average daily profiles | 24 hours (kWh) | k-means, DTW ±2 h | ___ | ___ | ___ |
| Statistics | mean, median, std | k-means | ___ | ___ | ___ |
| Statistics + domain hints | + min, max, neg_share | k-means | ___ | ___ | ___ |
| Statistics + domain hints | + min, max, neg_share | GMM | ___ | ___ | ___ |

Read the table from top to bottom. Normalising the profiles made the ARI slightly worse because size carries information in this data, DTW helped only for one of three random seeds, and replacing 24 hourly values by three yearly statistics barely changed it. The ARI only moved when the features started to reflect physics: `min` and `neg_share` respond to export and `max` to large loads. The last and largest gain came from the model rather than the features: a nearest-centre rule cannot hold a class as spread out as `EV_HP`, and a GMM can.

None of the label-free tools would have told us this. The elbow and silhouette did not point to the four classes, and PCA and t-SNE showed structure without measuring it. The ARI could, only because the labels were set aside; a DSO without labels is left with the cluster centres and the judgement of whether they make physical sense.

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

### Going further: more advanced features

</div>

The `PV` / `EV_PV` confusion survives every step above. The paper combines two families of features that go further than yearly statistics:

* **Domain-oriented**: one feature per DER, built so that the other DERs cannot mask it.
  * PV: `export_share` = exported / (imported + exported) energy.
  * EV: share of days with at least one hour above 3 kWh, or with ≥ 2 consecutive night hours above 3 kWh (a charging block); at night, PV cannot hide the charging.
  * Heat pump: winter / summer import using night hours only, so that PV does not inflate the ratio.
  * Keep at most one size feature (e.g. mean import): with several, k-means counts size several times.
* **Domain-agnostic** (time-series features, as in the paper): autocorrelation at lag 1 h, 24 h, sum of the first 10 lags (`acf1`, `acf10`), seasonal strength, entropy, trend. Libraries: [tsfeatures](https://github.com/Nixtla/tsfeatures), [IFEEL](https://maomaohu.net/software/ifeel/).